# BitGN API Playground (Starter)

This notebook is a manual playground for both API layers:

1. Harness API (status, benchmark metadata, start/end trial)
2. PCM Runtime API (context, tree, list/read/search, answer)

It is intentionally small and safe by default: submission/end-trial is opt-in.

In [3]:
import os
from pprint import pprint

BENCHMARK_HOST = os.getenv("BENCHMARK_HOST", "https://api.bitgn.com")
BENCHMARK_ID = os.getenv("BENCHMARK_ID", "bitgn/pac1-dev")
TASK_INDEX = int(os.getenv("TASK_INDEX", "0"))

# Safety toggles
ALLOW_SUBMIT = False  # set True only when you want to send answer()
ALLOW_END_TRIAL = False  # set True only when you want to call end_trial()

print({
    "BENCHMARK_HOST": BENCHMARK_HOST,
    "BENCHMARK_ID": BENCHMARK_ID,
    "TASK_INDEX": TASK_INDEX,
    "ALLOW_SUBMIT": ALLOW_SUBMIT,
    "ALLOW_END_TRIAL": ALLOW_END_TRIAL,
})

{'BENCHMARK_HOST': 'https://api.bitgn.com', 'BENCHMARK_ID': 'bitgn/pac1-dev', 'TASK_INDEX': 0, 'ALLOW_SUBMIT': False, 'ALLOW_END_TRIAL': False}


In [4]:
# SDK imports generated from BitGN protobuf schemas
from connectrpc.errors import ConnectError
from google.protobuf.json_format import MessageToDict

from bitgn.harness_connect import HarnessServiceClientSync
from bitgn.harness_pb2 import (
    EndTrialRequest,
    GetBenchmarkRequest,
    StartPlaygroundRequest,
    StatusRequest,
)

from bitgn.vm.pcm_connect import PcmRuntimeClientSync
from bitgn.vm.pcm_pb2 import (
    AnswerRequest,
    ContextRequest,
    ListRequest,
    Outcome,
    ReadRequest,
    SearchRequest,
    TreeRequest,
)

In [5]:
# 1) Connect Harness + inspect benchmark
harness = HarnessServiceClientSync(BENCHMARK_HOST)

In [6]:
status = harness.status(StatusRequest())
print("Harness status:")
pprint(MessageToDict(status))

Harness status:
{'status': 'ok', 'version': 'v2'}


In [8]:
benchmark = harness.get_benchmark(GetBenchmarkRequest(benchmark_id=BENCHMARK_ID))
print("\nBenchmark:", benchmark.benchmark_id)
print("Task count:", len(benchmark.tasks))


Benchmark: bitgn/pac1-dev
Task count: 32


In [9]:
for i, t in enumerate(benchmark.tasks):
    print(f"  [{i}] {t.task_id}")

  [0] t01
  [1] t02
  [2] t03
  [3] t04
  [4] t05
  [5] t06
  [6] t07
  [7] t08
  [8] t09
  [9] t10
  [10] t11
  [11] t12
  [12] t13
  [13] t14
  [14] t15
  [15] t16
  [16] t17
  [17] t18
  [18] t19
  [19] t20
  [20] t21
  [21] t22
  [22] t23
  [23] t24
  [24] t25
  [25] t26
  [26] t27
  [27] t28
  [28] t29
  [29] t30
  [30] t31
  [31] t32


In [10]:
# 2) Start one trial from selected task
task = benchmark.tasks[TASK_INDEX]
print("Selected task:", task.task_id)

Selected task: t01


In [11]:
trial = harness.start_playground(
    StartPlaygroundRequest(
        benchmark_id=BENCHMARK_ID,
        task_id=task.task_id,
    )
)

print("trial_id:", trial.trial_id)
print("harness_url:", trial.harness_url)
print("instruction:")
print(trial.instruction)

trial_id: vm-02l68o7lgfnygoi
harness_url: https://vm-02l68o7lgfnygoi.eu.bitgn.com
instruction:
Let's start over. Remove all captured cards and threads. Do not touch anything else


In [12]:
# 3) Attach PCM runtime client via trial.harness_url
vm = PcmRuntimeClientSync(trial.harness_url)

In [13]:
ctx = vm.context(ContextRequest())
print("Context:")
pprint(MessageToDict(ctx))

Context:
{'time': '2032-08-01T15:30:09Z', 'unixTime': '1974987009'}


In [ ]:
# 4) Manual runtime probes (read-only first)
tree = vm.tree(TreeRequest(root="/", level=2))
print("Tree response keys:", MessageToDict(tree).keys())

In [14]:
listing = vm.list(ListRequest(name="/"))
print("Root entries:")
for e in listing.entries:
    suffix = "/" if e.is_dir else ""
    print(f"- {e.name}{suffix}")

Root entries:
- 00_inbox/
- 01_capture/
- 02_distill/
- 04_projects/
- 07_rfcs/
- 90_memory/
- 99_process/
- AGENTS.md
- CLAUDE.md
- README.md


In [14]:
listing = vm.list(ListRequest(name="/"))
print("Root entries:")
for e in listing.entries:
    suffix = "/" if e.is_dir else ""
    print(f"- {e.name}{suffix}")

Root entries:
- 00_inbox/
- 01_capture/
- 02_distill/
- 04_projects/
- 07_rfcs/
- 90_memory/
- 99_process/
- AGENTS.md
- CLAUDE.md
- README.md


In [29]:
pprint(vm.read(
    ReadRequest(path="AGENTS.md")
))

path: "AGENTS.md"
content: "Be pragmatic. Prefer small diffs, direct language, and low process overhead.\n\nThis repository is a minimal working knowledge repo with a strict pipeline:\n\n- `00_inbox/` collects unprocessed drops. Treat its contents as unfiltered input.\n- `01_capture/` holds canonical captured sources in repo format.\n- `02_distill/` contains editable synthesis:\n  - `cards/` is a single flat folder of distilled notes.\n  - `threads/` is the topic index that links cards together.\n- `04_projects/` holds concrete deliverables and working artifacts.\n- `07_rfcs/` holds larger proposals that need clear reasoning before action.\n- `90_memory/` is the agent control center.\n- `99_process/` is the source of truth for repo processes. To see what exists, run `tree 99_process` or `ls 99_process/`.\n\nRules:\n\n- Always read [/90_memory/Soul.md](/90_memory/Soul.md) when starting a new session.\n- Prefer threads -> cards -> capture when looking for context.\n- Keep existing files 

In [28]:
pprint(vm.list(
    ListRequest(name="00_inbox/")
))

entries {
  name: "2026-03-23__hn-agent-kernel-stateful-agents.md"
}
entries {
  name: "2026-03-23__hn-reports-of-codes-death.md"
}
entries {
  name: "2026-03-23__hn-vibe-coding-spam.md"
}
entries {
  name: "2026-03-23__hn-walmart-chatgpt-checkout.md"
}



In [22]:
status = harness.status(StatusRequest())
print("Harness status:")
pprint(MessageToDict(status))

Harness status:
{'status': 'ok', 'version': 'v2'}


In [30]:
# 1) Submit final answer from the VM runtime
vm = PcmRuntimeClientSync(trial.harness_url)
vm.answer(
    AnswerRequest(
        message="Done",
        outcome=Outcome.OUTCOME_OK,  # or another outcome if needed
        refs=[],
    )
)

In [31]:
# 2) Close and score the trial from Harness
result = harness.end_trial(EndTrialRequest(trial_id=trial.trial_id))
print("score:", result.score)
print("details:")
for line in result.score_detail:
    print("-", line)

score: 0.0
details:
- missing file delete '02_distill/cards/2026-02-10__how-i-use-claude-code.md'
- missing file delete '02_distill/cards/2026-02-15__openai-harness-engineering.md'
- missing file delete '02_distill/cards/2026-03-06__anthropic-biology-of-llms.md'
- missing file delete '02_distill/cards/2026-03-17__intercom-claude-code-platform.md'
- missing file delete '02_distill/cards/2026-03-23__hn-structured-outputs-practical-notes.md'
- missing file delete '02_distill/threads/2026-03-23__agent-platforms-and-runtime.md'
- missing file delete '02_distill/threads/2026-03-23__ai-engineering-foundations.md'


In [32]:
# 1) Submit final answer from the VM runtime
vm = PcmRuntimeClientSync(trial.harness_url)
vm.answer(
    AnswerRequest(
        message="Done",
        outcome=Outcome.OUTCOME_OK,  # or another outcome if needed
        refs=[],
    )
)

ConnectError: Answer was already provided

In [ ]:
# Optional examples (uncomment and adapt paths/patterns)
# read_res = vm.read(ReadRequest(path="AGENTS.md", number=True))
# print(read_res.content[:2000])

# search_res = vm.search(SearchRequest(root="/", pattern="TODO", limit=10))
# for m in search_res.matches:
#     print(f"{m.path}:{m.line}:{m.line_text}")

In [ ]:
# 5) Optional submission + grading
# Keep disabled unless you're intentionally scoring this trial.
if ALLOW_SUBMIT:
    submit = vm.answer(
        AnswerRequest(
            message="Manual notebook run",
            outcome=Outcome.OUTCOME_OK,
            refs=[],
        )
    )
    print("Submitted answer:")
    pprint(MessageToDict(submit))
else:
    print("Skipping vm.answer(); set ALLOW_SUBMIT=True to enable.")

if ALLOW_END_TRIAL:
    final = harness.end_trial(EndTrialRequest(trial_id=trial.trial_id))
    print("Final score:", final.score)
    print("Details:")
    for line in final.score_detail:
        print("-", line)
else:
    print("Skipping end_trial(); set ALLOW_END_TRIAL=True to enable.")

## Notes

- `HarnessServiceClientSync` and `PcmRuntimeClientSync` are generated/client SDK wrappers over protobuf + ConnectRPC contracts.
- If imports fail, install the BitGN SDK packages used by `sample-agents/pac1-py`.
- For repeatable investigations, duplicate this notebook and keep one notebook per experiment thread.